In [134]:
import pandas as pd
import json

In [135]:
with open('../election-stations-2569.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# หาหน่วยเลือกตั้งของจังหวัด
rayong = next((p for p in data['provinces'] if p['name'] == 'ระยอง'), None)
# print(f"ระยอง {rayong['total_stations']} หน่วยเลือกตั้ง")
d = dict()
sd2d = dict()
t = 0
# หาหน่วยเลือกตั้งจากรหัส
def find_total_station(area,province):
    for p in data['provinces']:
        if p['name'] == province:
            for a in p['areas']:
                if a['area'] == area:
                    for unit in a['stations']:
                        if unit['district'] not in d:
                            d[unit['district']] = dict()
                        if unit['subdistrict'] not in  d[unit['district']]:
                            d[unit['district']][unit['subdistrict']] = 1
                            sd2d[unit['subdistrict']] = unit['district']
                        else:
                            d[unit['district']][unit['subdistrict']] += 1
    return None

find_total_station(2, 'ลำปาง')
print(d)
print(sd2d)
for k, v in d.items():
    for e, g in v.items():
        t+=g
print(t)

{'เมืองลำปาง': {'บ้านแลง': 12, 'บ้านเสด็จ': 19}, 'งาว': {'หลวงเหนือ': 7, 'หลวงใต้': 8, 'บ้านโป่ง': 12, 'บ้านร้อง': 13, 'ปงเตา': 13, 'นาแก': 6, 'บ้านอ้อน': 8, 'บ้านแหง': 9, 'บ้านหวด': 7, 'แม่ตีบ': 7}, 'แจ้ห่ม': {'แจ้ห่ม': 14, 'บ้านสา': 10, 'ปงดอน': 9, 'แม่สุก': 12, 'เมืองมาย': 7, 'ทุ่งผึ้ง': 8, 'วิเชตนคร': 12}, 'วังเหนือ': {'ทุ่งฮั้ว': 12, 'วังเหนือ': 12, 'วังใต้': 7, 'ร่องเคาะ': 18, 'วังทอง': 9, 'วังซ้าย': 10, 'วังแก้ว': 7, 'วังทรายคำ': 9}, 'เมืองปาน': {'เมืองปาน': 11, 'บ้านขอ': 14, 'ทุ่งกว๋าว': 16, 'แจ้ซ้อน': 15, 'หัวเมือง': 9}}
{'บ้านแลง': 'เมืองลำปาง', 'บ้านเสด็จ': 'เมืองลำปาง', 'หลวงเหนือ': 'งาว', 'หลวงใต้': 'งาว', 'บ้านโป่ง': 'งาว', 'บ้านร้อง': 'งาว', 'ปงเตา': 'งาว', 'นาแก': 'งาว', 'บ้านอ้อน': 'งาว', 'บ้านแหง': 'งาว', 'บ้านหวด': 'งาว', 'แม่ตีบ': 'งาว', 'แจ้ห่ม': 'แจ้ห่ม', 'บ้านสา': 'แจ้ห่ม', 'ปงดอน': 'แจ้ห่ม', 'แม่สุก': 'แจ้ห่ม', 'เมืองมาย': 'แจ้ห่ม', 'ทุ่งผึ้ง': 'แจ้ห่ม', 'วิเชตนคร': 'แจ้ห่ม', 'ทุ่งฮั้ว': 'วังเหนือ', 'วังเหนือ': 'วังเหนือ', 'วังใต้': 'วังเหนือ', 'ร่องเคาะ': 'วังเ

In [136]:
d['นอกเขต'] = dict()
for i in range(1,14):
    sub_dist = f"ชุดที่ {i}"
    sd2d[sub_dist] = 'นอกเขต'
    d['นอกเขต'][sub_dist] = 1
    # all_subdistrict.append(sub_dist)

In [137]:
d['วังเหนือ']['บ้านใหม่'] = 4
d['วังเหนือ']['วังเหนือ'] = 8
d['แจ้ห่ม']['แจ้ห่ม(ในเขต)'] = 6
d['แจ้ห่ม']['แจ้ห่ม(นอกเขต)'] = 8
d['แจ้ห่ม'].pop('แจ้ห่ม', None)
sd2d['บ้านใหม่'] = 'วังเหนือ'
sd2d['แจ้ห่ม(ในเขต)'] = "แจ้ห่ม"
sd2d['แจ้ห่ม(นอกเขต)'] = "แจ้ห่ม"

In [138]:
all_distrcit = []
all_subdistrict = []
district_count = {}
for k,v in d.items():
    all_distrcit.append(k)
    all_subdistrict += v.keys()
    for key in v.keys():
        if k not in district_count:
            district_count[k] = v[key]
        else: district_count[k]+= v[key]

In [139]:
for i in range(1,14):
    sub_dist = f"ชุดที่ {i}"
    all_subdistrict.append(sub_dist)

In [140]:
df = pd.read_csv("../ocr-result1/master_results.csv")

In [141]:
wrong_districts = []
for wrong_district in df['district'].unique().tolist():
    if wrong_district not in all_distrcit and wrong_district != 'นอกเขต':
        wrong_districts.append(wrong_district)

wrong_districts

['อำเภองาว',
 'อำเภอวังเหนือ',
 'อำเภอเมืองปาน',
 'อำเภอเมืองลำปาง (ต.บ้านแลง ต.บ้านเสด็จ)',
 'อำเภอแจ้ห่ม']

In [142]:
from rapidfuzz import process
def automate_edit(feature, check, wrongs, df):
    wrong2correct = dict()
    scores = dict()
    lis = []
    for text in wrongs:
        match = process.extractOne(text, check, score_cutoff=50)

        if match:
            result, score, index = match
            wrong2correct[text] = result
            scores[text] = round(score,2)
            if scores[text] >= 0.8:
                df.loc[df[feature] == text, feature] = result
            else:
                lis.append("text")
            print(f"OCR: '{text}' -> Corrected: '{result}' (Confidence: {score:.2f}%)")
        else:
            wrong2correct[text] = "can't find"
            scores[text] = 0
            lis.append(text)
            print(f"OCR: '{text}' -> [Manual Review Needed]")
    return wrong2correct, scores, lis

In [143]:
df_test = df.copy()
wrong2correct, scores, list_wrongs = automate_edit('district', all_distrcit, wrong_districts, df_test)

OCR: 'อำเภองาว' -> Corrected: 'งาว' (Confidence: 90.00%)
OCR: 'อำเภอวังเหนือ' -> Corrected: 'วังเหนือ' (Confidence: 90.00%)
OCR: 'อำเภอเมืองปาน' -> Corrected: 'เมืองปาน' (Confidence: 90.00%)
OCR: 'อำเภอเมืองลำปาง (ต.บ้านแลง ต.บ้านเสด็จ)' -> Corrected: 'เมืองลำปาง' (Confidence: 90.00%)
OCR: 'อำเภอแจ้ห่ม' -> Corrected: 'แจ้ห่ม' (Confidence: 90.00%)


In [144]:
wrong_sds = []
for wrong_sd in df_test['sub-district'].unique().tolist():
    if wrong_sd not in all_subdistrict:
        wrong_sds.append(wrong_sd)

wrong_sds

['เเจ้ซ้อน']

In [145]:
df_test.loc[df_test['sub-district'] == 'เเจ้ซ้อน', 'sub-district'] = 'แจ้ซ้อน'

In [146]:
l = df_test['sub-district'].unique().tolist()
for sd in all_subdistrict:
    if sd not in l:
        print(f"เขต {sd} หายไปทั้ง บช และ เขต")

In [147]:
df = df_test.copy()

In [148]:
correct_names = [
    "นางสาววิชุดา ว่องวัฒนวิโรจน์",
    "นางสาวสุวิภา กุศลจูง",
    "นายศรีพรหม หอมยก",
    "นายสมบูรณ์ รูปสะอาด",
    "นายดาชัย เอกปฐพี",
    "นายธนาธร โล่ห์สุนทร",
    "พันเอกสันทัด ภัทรกิตตินนท์",
]

In [150]:
party2number = dict()
with open("../partys_lists_number.txt", 'r', encoding="utf-8") as f:
    lines = []
    for line in f:
        if line.strip() == "":continue
        num, party = line.strip().split(":")
        num = int(num.strip().split()[1])
        party = party.strip()
        lines.append(party)
        party2number[party] = num
for i in range(1,8):
    party2number[correct_names[i-1]] = i
correct_names += lines

In [ ]:
df_test = df.copy()

In [ ]:
wrong_names = []
for wrong_name in df_test['name'].unique().tolist():
    if wrong_name not in all_subdistrict:
        wrong_names.append(wrong_name)

wrong_names

['ไทยทรัพย์ทวี',
 'เพื่อชาติไทย',
 'ใหม่',
 'มิติใหม่',
 'รวมใจไทย',
 'รวมไทยสร้างชาติ',
 'พลวัต',
 'ประชาธิปไตยใหม่',
 'เพื่อไทย',
 'ทางเลือกใหม่',
 'เศรษฐกิจ',
 'เสรีรวมไทย',
 'รวมพลังประชาชน',
 'ท้องที่ไทย',
 'อนาคตไทย',
 'พลังเพื่อไทย',
 'ไทยชนะ',
 'พลังสังคมใหม่',
 'สังคมประชาธิปไตยไทย',
 'พิวซัน',
 'ไทรวมพลัง',
 'ก้าวล่วง',
 'ปวงชนไทย',
 'วิชชันใหม่',
 'เพื่อชีวิตใหม่',
 'คลองไทย',
 'ประชาธิปัตย์',
 'ไทยก้าวหน้า',
 'ไทยภักดี',
 'แรงงานสร้างชาติ',
 'ประชากรไทย',
 'ครูไทยเพื่อประชาชน',
 'ประชาชาติ',
 'สรางอนาคตไทย',
 'รักชาติ',
 'ไทยพร้อม',
 'ภูมิใจไทย',
 'พลังธรรมใหม่',
 'กรีน',
 'ไทยธรรม',
 'แผ่นดินธรรม',
 'กล้าธรรม',
 'พลังประชารัฐ',
 'โอกาสใหม่',
 'เป็นธรรม',
 'ประชาชน',
 'ประชาไทย',
 'ไทยสร้างไทย',
 'ไทยกาวใหม่',
 'ประชาอาสาชาติ',
 'พร้อม',
 'เครือข่ายชาวนาแห่งประเทศไทย',
 'ไทยพิทักษ์ธรรม',
 'ความหวังใหม่',
 'ไทยรวมไทย',
 'เพื่อบ้านเมือง',
 'พลังไทยรักชาติ',
 'น.ส. วิชดา ว่องอัฒนาโรจน์',
 'น.ส. ศุภิกา กุศลจรูญ',
 'นายศรีพรหม พลอยยา',
 'นายสมบูรณ์ อุปมาธาตุ',
 'นายอาทิตย์ เอกปร

In [ ]:
wrong2correct, scores, list_wrongs = automate_edit('name', correct_names, wrong_names, df_test)

OCR: 'ไทยทรัพย์ทวี' -> Corrected: 'พรรคไทยทรัพย์ทวี' (Confidence: 85.71%)
OCR: 'เพื่อชาติไทย' -> Corrected: 'พรรคเพื่อชาติไทย' (Confidence: 85.71%)
OCR: 'ใหม่' -> Corrected: 'พรรคใหม่' (Confidence: 90.00%)
OCR: 'มิติใหม่' -> Corrected: 'พรรคมิติใหม่' (Confidence: 90.00%)
OCR: 'รวมใจไทย' -> Corrected: 'พรรครวมใจไทย' (Confidence: 90.00%)
OCR: 'รวมไทยสร้างชาติ' -> Corrected: 'พรรครวมไทยสร้างชาติ' (Confidence: 88.24%)
OCR: 'พลวัต' -> Corrected: 'พรรคพลวัต' (Confidence: 90.00%)
OCR: 'ประชาธิปไตยใหม่' -> Corrected: 'พรรคประชาธิปไตยใหม่' (Confidence: 88.24%)
OCR: 'เพื่อไทย' -> Corrected: 'พรรคเพื่อไทย' (Confidence: 90.00%)
OCR: 'ทางเลือกใหม่' -> Corrected: 'พรรคทางเลือกใหม่' (Confidence: 85.71%)
OCR: 'เศรษฐกิจ' -> Corrected: 'พรรคเศรษฐกิจ' (Confidence: 90.00%)
OCR: 'เสรีรวมไทย' -> Corrected: 'พรรคเสรีรวมไทย' (Confidence: 83.33%)
OCR: 'รวมพลังประชาชน' -> Corrected: 'พรรครวมพลังประชาชน' (Confidence: 87.50%)
OCR: 'ท้องที่ไทย' -> Corrected: 'พรรคท้องที่ไทย' (Confidence: 83.33%)
OCR: 'อนาคตไทย' ->

In [ ]:
list_wrongs

['บริษัทโฮลดิง',
 'รวมหลังหักภาษี',
 'คืนเพิ่มเติม',
 'โดยสาร',
 'สัญญาเช่าซื้อก่อนกำหนด',
 'โทรคมนาคม',
 nan,
 '-1',
 'รวมคะแนนทั้งสิ้น',
 'กรุงเทพมหานคร',
 'พิมพ์ขึ้น',
 'โทรสารเคล็ง',
 'พิกุลชน',
 'ฟิลิปปิน',
 'ไทกรุงเทพฯ']

In [ ]:
for wrong in list_wrongs:
    print(df[df['name'] == wrong])

     type province district sub-district  unit_number          name score
2740   บช    ลำปาง      งาว      บ้านหวด            5  บริษัทโฮลดิง     1
     type province district sub-district  unit_number            name score
2741   บช    ลำปาง      งาว      บ้านหวด            5  รวมหลังหักภาษี     6
     type province district sub-district  unit_number          name score
2742   บช    ลำปาง      งาว      บ้านหวด            5  คืนเพิ่มเติม     1
     type province district sub-district  unit_number    name score
2745   บช    ลำปาง      งาว      บ้านหวด            5  โดยสาร     3
     type province district sub-district  unit_number                    name  \
2747   บช    ลำปาง      งาว      บ้านหวด            5  สัญญาเช่าซื้อก่อนกำหนด   

     score  
2747     0  
     type province district sub-district  unit_number       name score
2749   บช    ลำปาง      งาว      บ้านหวด            5  โทรคมนาคม     0
Empty DataFrame
Columns: [type, province, district, sub-district, unit_number, name, 

In [ ]:
df_test['number_party'] = df_test['name'].apply()

{'พรรคไทยทรัพย์ทวี': 1,
 'พรรคเพื่อชาติไทย': 2,
 'พรรคใหม่': 3,
 'พรรคมิติใหม่': 4,
 'พรรครวมใจไทย': 5,
 'พรรครวมไทยสร้างชาติ': 6,
 'พรรคพลวัต': 7,
 'พรรคประชาธิปไตยใหม่': 8,
 'พรรคเพื่อไทย': 9,
 'พรรคทางเลือกใหม่': 10,
 'พรรคเศรษฐกิจ': 11,
 'พรรคเสรีรวมไทย': 12,
 'พรรครวมพลังประชาชน': 13,
 'พรรคท้องที่ไทย': 14,
 'พรรคอนาคตไทย': 15,
 'พรรคพลังเพื่อไทย': 16,
 'พรรคไทยชนะ': 17,
 'พรรคพลังสังคมใหม่': 18,
 'พรรคสังคมประชาธิปไตย': 19,
 'พรรคฟิวชัน': 20,
 'พรรคไทรวมพลัง': 21,
 'พรรคก้าวอิสระ': 22,
 'พรรคปวงชนไทย': 23,
 'พรรควิชชั่นใหม่': 24,
 'พรรคเพื่อชีวิตใหม่': 25,
 'พรรคคลองไทย': 26,
 'พรรคประชาธิปัตย์': 27,
 'พรรคไทยก้าวหน้า': 28,
 'พรรคไทยภักดี': 29,
 'พรรคแรงงานสร้างชาติ': 30,
 'พรรคประชากรไทย': 31,
 'พรรคครูไทยเพื่อประชาชน': 32,
 'พรรคประชาชาติ': 33,
 'พรรคสร้างอนาคตไทย': 34,
 'พรรครักชาติ': 35,
 'พรรคไทยพร้อม': 36,
 'พรรคภูมิใจไทย': 37,
 'พรรคพลังธรรมใหม่': 38,
 'พรรคกรีน': 39,
 'พรรคไทยธรรม': 40,
 'พรรคแผ่นดินธรรม': 41,
 'พรรคกล้าธรรม': 42,
 'พรรคพลังประชารัฐ': 43,
 'พรรคโอกาสใหม่'